## Claude Sonnet 4
Claude Sonnet 4 är Anthropics mest avancerade modell med förbättrad kontextförståelse, resonemang och högre kvalitet i strukturerad extraktion jämfört med tidigare versioner. Vi kommer att utvärdera dessa förbättringar i vår testning för att se hur modellen presterar på våra PDF-analyser.

In [5]:
!pip install openai pymupdf python-dotenv pillow --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 86.0 MB/s eta 0:00:00


### API Key

In [2]:
import os

def load_api_key() -> str:
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            print("API-nyckel laddad från Colab Secrets")
            return key
    except (ImportError, Exception):
        pass

    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    key = os.environ.get("OPENROUTER_API_KEY")
    if key:
        print("API-nyckel laddad från miljövariabel / .env")
        return key

os.environ["OPENROUTER_API_KEY"] = load_api_key()


API-nyckel laddad från Colab Secrets


### Load PDF

In [3]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
except ImportError:
    pdf_path = "data/benchmark"

print("Using PDF:", pdf_path)

Saving FlexQube.pdf to FlexQube.pdf
Using PDF: FlexQube.pdf


### PDF -> images -> Claude Sonnet 4

In [10]:
import fitz
import base64
import io
import time
from pathlib import Path
from PIL import Image
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODEL = "anthropic/claude-sonnet-4"

MAX_IMAGE_BYTES = 3_500_000  # 3.5MB raw → ~4.7MB base64, safely under 5MB


def pdf_to_images(pdf_path: str, dpi: int = 150) -> list[Image.Image]:
    doc = fitz.open(pdf_path)
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat)
        images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
    doc.close()
    return images


def image_to_base64(image: Image.Image) -> tuple[str, str]:
    for quality in [85, 65, 45, 30]:
        buf = io.BytesIO()
        image.save(buf, format="JPEG", quality=quality)
        if buf.tell() <= MAX_IMAGE_BYTES:
            return base64.b64encode(buf.getvalue()).decode(), "image/jpeg"

    img = image
    for scale in [0.7, 0.5, 0.3]:
        img = image.resize(
            (int(image.width * scale), int(image.height * scale)),
            Image.LANCZOS,
        )
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=40)
        if buf.tell() <= MAX_IMAGE_BYTES:
            return base64.b64encode(buf.getvalue()).decode(), "image/jpeg"

    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=20)
    return base64.b64encode(buf.getvalue()).decode(), "image/jpeg"


def process_page(image: Image.Image) -> str:
    b64, mime_type = image_to_base64(image)

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=8000,
        temperature=0,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Extract all text from this document image. "
                        "Return ONLY clean Markdown preserving structure "
                        "(headings, lists, tables). No commentary."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{mime_type};base64,{b64}"
                    },
                },
            ],
        }],
    )
    return response.choices[0].message.content

### Run OCR

In [11]:
pdf_files = []
p = Path(pdf_path)

if p.is_dir():
    pdf_files = sorted(p.glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(f"No PDFs found in {p}")
    print(f"Found {len(pdf_files)} PDF(s) in {p}")
else:
    pdf_files = [p]

all_results = {}

for pdf_file in pdf_files:
    print(f"\n{'='*50}")
    print(f"PDF: {pdf_file.name}")
    print(f"{'='*50}")

    images = pdf_to_images(str(pdf_file))
    print(f"Pages: {len(images)}")

    pages = []
    start = time.time()

    for i, img in enumerate(images):
        print(f"  Page {i+1}/{len(images)}...", end=" ", flush=True)
        t0 = time.time()
        text = process_page(img)
        print(f"({time.time()-t0:.1f}s)")
        pages.append(f"# Page {i+1}\n\n{text}")

    elapsed = time.time() - start
    full_md = "\n\n---\n\n".join(pages)
    all_results[pdf_file.name] = full_md

    print(f"Done: {len(full_md):,} chars in {elapsed:.1f}s")



PDF: FlexQube.pdf
MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

Pages: 68
  Page 1/68... (2.9s)
  Page 2/68... (15.9s)
  Page 3/68... (2.9s)
  Page 4/68... (30.1s)
  Page 5/68... (17.2s)
  Page 6/68... (25.7s)
  Page 7/68... (26.8s)
  Page 8/68... (18.6s)
  Page 9/68... (9.6s)
  Page 10/68... (23.6s)
  Page 11/68... (11.7s)
  Page 12/68... (21.3s)
  Page 13/68... (25.6s)
  Page 14/68... (38.7s)
  Page 15/68... (5.1s)
  Page 16/68... (15.4s)
  Page 17/68... (18.6s)
  Page 18/68... (21.7s)
  Page 19/68... (15.5s)
  Page 20/68... (18.5s)
  Page 21/68... (24.0s)
  Page 22/68... (4.4s)
  Page 23/68... (22.1s)
  Page 24/68... (29.1s)
  Page 25/68... (39.7s)
  Page 26/68... (12.2s)
  Page 27/68... (14.2s)
  Page 28/68... (20.8s)
  Page 29/68... (14.1s)
  Page 30/68... (38.9s)
  Page 31/68... (14.2s)
  Page 32/68... (55.9s)
  Page 33/68... (22.4s

In [12]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

for pdf_name, md_text in all_results.items():
    pdf_stem = Path(pdf_name).stem
    out_path = output_dir / f"claude_sonnet4_{pdf_stem}.md"
    out_path.write_text(md_text, encoding="utf-8")
    print(f"Saved: {out_path}")

    try:
        from google.colab import files as colab_files
        colab_files.download(str(out_path))
    except ImportError:
        pass

Saved: output/claude_sonnet4_FlexQube.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
first_md = list(all_results.values())[0]
print(first_md[:2000])
if len(first_md) > 2000:
    print("\n...")

# Page 1

# Årsredovisning 2022

![FLEXQUBE logo]

---

# Page 2

# Det här är FlexQube

FlexQube är en global leverantör av modulära och robusta mekaniska vagnar och robotiserade lösningar för materialhantering. Koncernen grundades 2010 och har sedan dess säkrat ett stort antal framstående företag som kunder.

FlexQube är ett teknikbolag med huvudkontor i Göteborg samt egna verksamheter i USA, Mexiko, Tyskland och England. Bolaget är verksamt inom vagnbaserad materialhantering genom ett patenterat modulkoncept. FlexQube utvecklar och designar kundanpassade lösningar för både robotiserad och mekaniserad materiallogistik. Genom företagets egenutvecklade och unika automationskoncept erbjuds robusta och självkörande robotvagnar. FlexQube har över 1000 kunder i 37 länder där de primära marknaderna är Nordamerika och Europa.

FlexQubes kunder återfinns inom bland annat tillverkningsindustrin, distributions- och lagerverksamheter. Exempel på större kunder är Tesla, Amazon, Volvo Cars, Siemen